First we look at the process model 


In [1]:
import control as ct
import matplotlib.pyplot as plt
import numpy as np

T = 10 # samplingsfrekvens, once per 10 seconds

z = ct.tf('z')
G = T/(1-z**(-1))
plt.figure(1)
ct.pzmap(G)
plt.title("Pole/zero plot for plant")
plt.figure(2)
ct.bode_plot(G, display_margins=True)
plt.title("Bode plot of plant")
plt.show()


In [2]:
hr = np.linspace(0, 50, 10000)
Gi = 1/( 1 - z**(-1))
ct.rlocus(G*Gi , gains=hr)
plt.title("Possible conjugate pairs of poles using I-controller")
plt.show()


Ignoring fixed x limits to fulfill fixed data aspect with adjustable data limits.


Now place the poles


In [3]:
Kp = 40
Ki = 50

Gpi = (Kp*(z-1) + Ki*z)/(z-1)
Gcl = ct.feedback(G*Gpi, 1)
print(Gcl)
poles = ct.poles(Gcl)
plt.figure(1)
ct.pzmap(Gcl)
plt.title("Pole/zero plot of PI-controller")
print(poles)
plt.figure(2)
ct.bode_plot(G*Gpi, display_margins=True)
plt.title("Bode plot of closed loop")
plt.show()


<TransferFunction>: sys[33]
Inputs (1): ['u[0]']
Outputs (1): ['y[0]']
dt = True
    900 z^2 - 400 z
  -------------------
  901 z^2 - 402 z + 1
[0.44366933+0.j 0.00250159+0.j]


Now find the Kp and Ki using pole placement


In [4]:
def design_discrete_pi(T, desired_poles):
    """
    Calculates Kp and Ki for a discrete PI controller 
    given a sampling time T and desired closed-loop poles.
    Assumes plant G(z) = (T*z) / (z-1)
    """
    desired_poly = np.poly(desired_poles)
    d1 = desired_poly[1]
    d0 = desired_poly[2]
    
    Kp = (-2 * d0 - d1) / (T * d0)
    Ki = (1 + d0 + d1) / (T * d0)
    
    return Kp, Ki


In [5]:
T = 10
desired_poles = [0.3 + 0.1j, 0.3 - 0.1j]
Kp_calc, Ki_calc = design_discrete_pi(T, desired_poles)

print(f"Calculated Kp: {Kp_calc:.4f}")
print(f"Calculated Ki: {Ki_calc:.4f}")

z = ct.tf('z')
G = T / (1 - z**(-1))
Gpi = (Kp_calc*(z-1) + Ki_calc*z) / (z-1)

Gcl = ct.feedback(G * Gpi, 1)

print(f"\nActual Closed-loop Poles for tau={T}:")
print(ct.poles(Gcl))

plt.figure(figsize=(6, 5))
ct.pzmap(Gcl)
plt.title(f"Correct Pole Placement via Diophantine")
plt.show()


Calculated Kp: 0.4000
Calculated Ki: 0.5000
Actual Closed-loop Poles for tau=10:
[0.3+0.1j 0.3-0.1j]


In [6]:
%matplotlib inline
import numpy as np
import control as ct
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

T = 10 
G = ct.tf([T, 0], [1, -1], T)
output = widgets.Output()

def update_plots(change=None):
    Kp = slider_Kp.value
    Ki = slider_Ki.value
    
    with output:
        clear_output(wait=True)
        
        # 1. Set up a clean layout for exactly ONE plot
        fig, ax_pz = plt.subplots(1, 1, figsize=(6, 6))
        
        # 2. Calculate discrete systems
        z = ct.TransferFunction.z
        z.dt = T
        Gpi = (Kp*(z-1) + Ki*z)/(z-1)
        Gcl = ct.feedback(G*Gpi, 1)
        
        # 3. Draw Pole/Zero Map
        ct.pzmap(Gcl, ax=ax_pz, title=f"Pole/Zero Plot (Kp={Kp}, Ki={Ki})")
        
        # Formatting to keep the unit circle perfectly round
        ax_pz.grid(True)
        ax_pz.set_aspect('equal', adjustable='box')
        ax_pz.set_xlim([-1.2, 1.2])                  
        ax_pz.set_ylim([-1.2, 1.2])
        
        # 4. Render directly to the Notebook Canvas
        fig.canvas.draw()
        display(fig)
        plt.close(fig)

# 3. Create Sliders
slider_Kp = widgets.FloatSlider(
    value=4.0, min=0.1, max=10.0, step=0.1, description="Kp:"
)
slider_Ki = widgets.FloatSlider(
    value=5.0, min=0.1, max=10.0, step=0.1, description="Ki:"
)

# Connect sliders to update function
slider_Kp.observe(update_plots, names="value")
slider_Ki.observe(update_plots, names="value")

# 4. Render Layout
ui = widgets.VBox([slider_Kp, slider_Ki, output])
update_plots()  # Run once initially

display(ui)


In [17]:
import sympy as sp
import numpy as np

def auto_place_pi(desired_poles):
    z, Kp, Ki, T = sp.symbols('z Kp Ki, T')
    
    # Define the Transfer Function components
    plant_num = T * z
    plant_den = z - 1
    pi_num = Kp * (z - 1) + Ki * z
    pi_den = z - 1
    
    # Build the Characteristic Polynomial: (Plant_Den * PI_Den) + (Plant_Num * PI_Num) = 0
    char_eq = sp.Poly(plant_den * pi_den + plant_num * pi_num, z)
    actual_coeffs = char_eq.all_coeffs()
    print("actual coeffs:")
    sp.pprint(actual_coeffs)
    
    # Get the target polynomial coefficients from the desired poles
    desired_poly = np.poly(desired_poles)
    
    # Set up equations matching Actual coefficients to Desired coefficients
    # (We divide by actual_coeffs[0] to normalize the polynomial so the highest power is 1)
    equations = []
    for actual, desired in zip(actual_coeffs, desired_poly):
        equations.append(sp.Eq(actual / actual_coeffs[0], desired))
        
    # Let SymPy solve the algebra
    solution = sp.solve(equations, (Kp, Ki), dict=True)[0]
    return (solution[Kp]), (solution[Ki])

T = 0.1
desired_poles = [0.3 + 0.1j, 0.3 - 0.1j]

Kp_calc, Ki_calc = auto_place_pi(desired_poles)

print(f"SymPy Calculated Kp: {Kp_calc}")
print(f"SymPy Calculated Ki: {Ki_calc}")


actual coeffs:
[Ki⋅T + Kp⋅T + 1, -Kp⋅T - 2, 1]
SymPy Calculated Kp: 4.0/T
SymPy Calculated Ki: 5.0/T


In [14]:

T = 0.3
desired_poles = [0.3 + 0.1j, 0.3 - 0.1j]

Kp_calc, Ki_calc = auto_place_pi(T, desired_poles)

print(f"SymPy Calculated Kp: {Kp_calc:.4f}")
print(f"SymPy Calculated Ki: {Ki_calc:.4f}")


actual coeffs:
0.3⋅Ki + 0.3⋅Kp + 1.0
─────────────────────
    -0.3⋅Kp - 2.0
SymPy Calculated Kp: 13.3333
SymPy Calculated Ki: 16.6667
